# Preprocess Amazon Reviews 2023

Ép Amazon Reviews 2023 về 4 file chuẩn cho pipeline:

- `interactions.parquet`: `user_idx, item_idx, timestamp, split`
- `item_lookup.parquet`: `item_idx, title, avg_rating, rating_number, categories`
- `user2idx.json`: raw `user_id` -> `user_idx`
- `item2idx.json`: raw `parent_asin` -> `item_idx`

Luồng: load + merge → k-core → gán idx → leave-one-out split → lưu.

In [ ]:
import sys
from pathlib import Path

# Thêm thư mục cha của `package/` vào sys.path (bỏ cell này nếu package đã được install)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "package").is_dir())
sys.path.insert(0, str(ROOT))

from package.config.data import AMAZON_PROCESSED_DIR, AMAZON_RAW_DIR
from package.preprocess import (
    assign_idx,
    kcore_filter,
    leave_one_out_split,
    load_canonical_table,
)
from package.utils._export import write_outputs

## Tham số

In [ ]:
CATEGORY = "Video_Games"
MIN_INTERACTIONS = 5

RAW_DIR = AMAZON_RAW_DIR
OUT_DIR = AMAZON_PROCESSED_DIR / CATEGORY
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"raw: {RAW_DIR}")
print(f"out: {OUT_DIR}")

## 1. Load + merge review với metadata

In [ ]:
canonical = load_canonical_table(CATEGORY, str(RAW_DIR))
print(f"{len(canonical)} dòng sau merge review+metadata")
canonical.head()

## 2. K-core

In [ ]:
canonical = kcore_filter(canonical, MIN_INTERACTIONS)
print(f"{len(canonical)} dòng sau k-core (>= {MIN_INTERACTIONS})")

## 3. Gán idx + leave-one-out split

In [ ]:
canonical, user2idx, item2idx = assign_idx(canonical)
canonical = leave_one_out_split(canonical)
print(f"{len(user2idx)} user, {len(item2idx)} item")
canonical["split"].value_counts()

## 4. Lưu 4 file

In [ ]:
write_outputs(canonical, user2idx, item2idx, str(OUT_DIR))
sorted(p.name for p in OUT_DIR.iterdir())